In [8]:
from collections import OrderedDict
from typing import Dict
from pathlib import Path
import torch
import torch.nn as nn

In [9]:
DROPOUT = 0.6
HIDDEN_DIMS = [512, 256, 128]
INPUT_DIM = 6144

In [10]:
class DermFoundationMLPHead(nn.Sequential):
    """
    Exact MLP head used after Derm Foundation embeddings.

    Architecture:
        Linear(input_dim, 512) -> ReLU -> Dropout(0.6)
        Linear(512, 256)      -> ReLU -> Dropout(0.6)
        Linear(256, 128)      -> ReLU -> Dropout(0.6)
        Linear(128, num_classes)
    """

    def __init__(self, input_dim: int, num_classes: int):
        super().__init__(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(DROPOUT),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(DROPOUT),

            nn.Linear(128, num_classes),
        )
        self.input_dim = input_dim
        self.output_dim = num_classes

In [11]:
def load_mlp_head(
    checkpoint_path: str | Path,
    num_classes: int,
    device: torch.device,
    input_dim: int = INPUT_DIM,
) -> nn.Module:
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"MLP checkpoint not found: {checkpoint_path}")

    model = DermFoundationMLPHead(
        input_dim=input_dim,
        num_classes=num_classes,
    ).to(device)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    state_dict = checkpoint["model_state_dict"]

    model.load_state_dict(state_dict, strict=True)
    model.eval()

    return model

In [12]:
model = load_mlp_head("derm_foundation_mlp_head.pt", num_classes=23, device=torch.device("cpu"))

In [13]:
print(model)

DermFoundationMLPHead(
  (0): Linear(in_features=6144, out_features=512, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.6, inplace=False)
  (3): Linear(in_features=512, out_features=256, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.6, inplace=False)
  (6): Linear(in_features=256, out_features=128, bias=True)
  (7): ReLU()
  (8): Dropout(p=0.6, inplace=False)
  (9): Linear(in_features=128, out_features=23, bias=True)
)
